# 00 — Document Audit, Metadata & Data Cleaning

## Objective
Prepare source documents for a reliable Retrieval-Augmented Generation (RAG) pipeline.

## Dataset groups
- Controlled development dataset: Northwind policy PDFs.
- Public evaluation dataset: genuine documents collected from official public sources.

## Responsibilities
1. Discover and inventory PDF files.
2. Extract raw text while preserving page references.
3. Extract document metadata.
4. Validate metadata and flag uncertainty.
5. Clean extracted text without changing its meaning.
6. Detect data-quality problems.
7. Export validated metadata, cleaned text, and quality reports.

## Important rules
- Preserve original PDFs unchanged.
- Do not invent missing metadata.
- Keep controlled and public datasets separate.
- Preserve document IDs, dates, amounts, tables, and policy clauses.
- Record validation failures rather than silently ignoring them.

In [ ]:
%cd /content
!git clone https://github.com/Adityasrivastava3000/secure-rag-platform-1.git

/content
Cloning into 'secure-rag-platform-1'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 98 (delta 31), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 112.08 KiB | 4.15 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [ ]:
from pathlib import Path

# This notebook runs in Google Colab.
PROJECT_ROOT = Path("/content/secure-rag-platform-1")

DATA_DIR = PROJECT_ROOT / "data"
CONTROLLED_DIR = DATA_DIR / "raw"
PUBLIC_DIR = DATA_DIR / "public_raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = DATA_DIR / "quality_reports"

print("Project root:", PROJECT_ROOT)
print("Repository exists:", PROJECT_ROOT.exists())

Project root: /content/secure-rag-platform-1
Repository exists: True


In [ ]:
pdf_files = sorted(CONTROLLED_DIR.rglob("*.pdf"))  #rglob('*.pdf') finds pdf inside all subfolders

print("Total controlled PDFs:", len(pdf_files))

for path in pdf_files:
    print(path.relative_to(PROJECT_ROOT))  # path.relative->prints cleaner paths relative to your repository

Total controlled PDFs: 13
data/raw/HR/HR-BEN-005_Employee_Benefits_v4.0.pdf
data/raw/HR/HR-COC-003_Code_of_Conduct_v5.0.pdf
data/raw/HR/HR-LV-001_Leave_Policy_v2.0.pdf
data/raw/HR/HR-LV-001_Leave_Policy_v3.2.pdf
data/raw/HR/HR-TRV-004_Travel_Policy_v4.1.pdf
data/raw/HR/HR-WFH-002_Work_From_Home_Policy_v2.1.pdf
data/raw/IT/IT-DEV-002_Device_Policy_v3.0.pdf
data/raw/IT/IT-IR-004_Incident_Response_v3.1.pdf
data/raw/IT/IT-PWD-001_Password_Policy_v2.4.pdf
data/raw/IT/IT-SEC-003_Security_Policy_v4.2.pdf
data/raw/finance/FIN-EXP-001_Expense_Policy_v3.3.pdf
data/raw/finance/FIN-PRC-003_Procurement_Policy_v4.0.pdf
data/raw/finance/FIN-RMB-002_Reimbursement_Policy_v2.0.pdf


In [ ]:
#pdf to text extractor and orgaize metadata into tables
%pip install -q pypdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 4.8 MB/s eta 0:00:00


checking is text can be extacted by the pdf

In [ ]:
from pypdf import PdfReader

pdf_path = pdf_files[0]
reader = PdfReader(str(pdf_path))

print("File:", pdf_path.name)
print("Pages:", len(reader.pages))

for page_number, page in enumerate(reader.pages[:2], start=1):
    text = page.extract_text() or ""
    print(f"\n--- Page {page_number} ---")
    print(text[:2500])

File: HR-BEN-005_Employee_Benefits_v4.0.pdf
Pages: 2

--- Page 1 ---
Northwind Technologies Pvt. Ltd. | HR-BEN-005 Employee Benefits v4.0 | Internal
Page 1
Employee Benefits
Northwind Technologies Pvt. Ltd. - Human Resources Policy
Document ID
Version
Effective Date
Status
HR-BEN-005
4.0
2025-07-01
Active
Owner
Approved By
Classification
Review Cycle
Head of Total Rewards
CHRO
Internal
Annual
1. Overview
 1.1  This document summarises the benefits available to permanent employees of Northwind
 Technologies. Where a detailed policy exists, that policy governs unless this document is stated to
be more recent and specific.
2. Health and Insurance
Benefit
Coverage
Group Health Insurance (employee)
INR 5,00,000 sum insured
Family floater (spouse and up to 2 children)
INR 10,00,000 sum insured, no additional premium
Parents / parents-in-law
Optional; employee pays premium
Group Term Life
3 times annual base salary
Personal Accident Cover
INR 25,00,000
3. Time Off and Flexibility
 3.1  Employ

inspecting text quality
of while 13 pdfs

In [ ]:
#inspect text quality
from pypdf import PdfReader
import pandas as pd

text_quality = []

for path in pdf_files:
    reader = PdfReader(str(path))
    page_texts = [
        page.extract_text() or ""
        for page in reader.pages
    ]

    full_text = "\n".join(page_texts)

    text_quality.append({
        "filename": path.name,
        "pages": len(reader.pages),
        "empty_pages": sum(not text.strip() for text in page_texts),
        "characters": len(full_text),
        "replacement_chars": full_text.count("\ufffd"),
        "has_text": bool(full_text.strip()),
    })

quality_df = pd.DataFrame(text_quality)
display(quality_df)

,filename,pages,empty_pages,characters,replacement_chars,has_text
0,HR-BEN-005_Employee_Benefits_v4.0.pdf,2,0,2249,0,True
1,HR-COC-003_Code_of_Conduct_v5.0.pdf,1,0,2138,0,True
2,HR-LV-001_Leave_Policy_v2.0.pdf,1,0,832,0,True
3,HR-LV-001_Leave_Policy_v3.2.pdf,1,0,2201,0,True
4,HR-TRV-004_Travel_Policy_v4.1.pdf,1,0,1777,0,True
5,HR-WFH-002_Work_From_Home_Policy_v2.1.pdf,1,0,1886,0,True
6,IT-DEV-002_Device_Policy_v3.0.pdf,1,0,1697,0,True
7,IT-IR-004_Incident_Response_v3.1.pdf,2,0,2474,0,True
8,IT-PWD-001_Password_Policy_v2.4.pdf,1,0,1513,0,True
9,IT-SEC-003_Security_Policy_v4.2.pdf,2,0,2553,0,True


checking metadata and one none value has been found
so we check the text of it's neccesary or to do be archived

In [ ]:
import re
import pandas as pd
from pypdf import PdfReader

metadata_records = []

for path in pdf_files:
    reader = PdfReader(str(path))
    text = "\n".join(page.extract_text() or "" for page in reader.pages)

    def extract(pattern):
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        return match.group(1).strip() if match else None

    metadata_records.append({
        "filename": path.name,
        "document_id": extract(r"(?m)^([A-Z]{2,5}-[A-Z]{2,5}-\d{3})\s*$"),
        "version": extract(r"(?m)^(\d+\.\d+)\s*$"),
        "effective_date": extract(r"(?m)^(\d{4}-\d{2}-\d{2})\s*$"),
        "status": extract(r"(?m)^(Active|Archived|Draft|Superseded)\s*$"),
        "classification": extract(
            r"(?m)^(Public|Internal|Confidential|Restricted)\s*$"
        ),
    })

metadata_df = pd.DataFrame(metadata_records)
display(metadata_df)


,filename,document_id,version,effective_date,status,classification
0,HR-BEN-005_Employee_Benefits_v4.0.pdf,HR-BEN-005,4.0,2025-07-01,Active,Internal
1,HR-COC-003_Code_of_Conduct_v5.0.pdf,HR-COC-003,5.0,2025-02-01,Active,Internal
2,HR-LV-001_Leave_Policy_v2.0.pdf,HR-LV-001,2.0,2023-04-01,None,Internal
3,HR-LV-001_Leave_Policy_v3.2.pdf,HR-LV-001,3.2,2025-04-01,Active,Internal
4,HR-TRV-004_Travel_Policy_v4.1.pdf,HR-TRV-004,4.1,2025-03-01,Active,Internal
5,HR-WFH-002_Work_From_Home_Policy_v2.1.pdf,HR-WFH-002,2.1,2025-01-15,Active,Internal
6,IT-DEV-002_Device_Policy_v3.0.pdf,IT-DEV-002,3.0,2025-02-15,Active,Internal
7,IT-IR-004_Incident_Response_v3.1.pdf,IT-IR-004,3.1,2025-06-01,Active,Confidential
8,IT-PWD-001_Password_Policy_v2.4.pdf,IT-PWD-001,2.4,2024-11-01,Active,Internal
9,IT-SEC-003_Security_Policy_v4.2.pdf,IT-SEC-003,4.2,2025-05-01,Active,Internal


In [ ]:
from pypdf import PdfReader

target = next(
    p for p in pdf_files
    if p.name == "HR-LV-001_Leave_Policy_v2.0.pdf"
)

reader = PdfReader(str(target))
text = "\n".join(page.extract_text() or "" for page in reader.pages)

print(text[:1800])

Northwind Technologies Pvt. Ltd. | HR-LV-001 Leave Policy v2.0 | Internal
Page 1
*** ARCHIVED (SUPERSEDED BY V3.2) - DO NOT USE ***
Leave Policy
Northwind Technologies Pvt. Ltd. - Human Resources Policy
Document ID
Version
Effective Date
Status
HR-LV-001
2.0
2023-04-01
Archived (Superseded by
v3.2)
Owner
Approved By
Classification
Review Cycle
Head of HR Operations
CHRO
Internal
Annual
1. Leave Entitlements
 Leave Type
Annual Entitlement
Earned Leave (EL)
15 working days
Casual Leave (CL)
7 working days
Sick Leave (SL)
8 working days
Paternity Leave
5 working days
2. Carry Forward
 2.1  A maximum of 5 unused EL days may be carried forward to the next year.
 2.2  Planned leave must be requested at least 3 days in advance.
 
3. Note
This version was superseded on 2025-04-01 by v3.2 and is retained for audit purposes only.



In [ ]:
import re
import pandas as pd
from pypdf import PdfReader

metadata_records = []

for path in pdf_files:
    reader = PdfReader(str(path))
    text = "\n".join(page.extract_text() or "" for page in reader.pages)

    def extract(pattern):
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        return match.group(1).strip() if match else None

    metadata_records.append({
        "filename": path.name,
        "document_id": extract(
            r"(?m)^([A-Z]{2,5}-[A-Z]{2,5}-\d{3})\s*$"
        ),
        "version": extract(
            r"(?m)^(\d+\.\d+)\s*$"
        ),
        "effective_date": extract(
            r"(?m)^(\d{4}-\d{2}-\d{2})\s*$"
        ),
        "status": extract(
            r"(?m)^(Active|Archived|Draft|Superseded)"
            r"(?:\s*\(.*)?$"
        ),
        "classification": extract(
            r"(?m)^(Public|Internal|Confidential|Restricted)\s*$"
        ),
    })

metadata_df = pd.DataFrame(metadata_records)

# Check for missing required metadata
required_fields = [
    "document_id",
    "version",
    "effective_date",
    "status",
    "classification",
]

display(metadata_df)

print("\nMissing values by field:")
display(metadata_df[required_fields].isna().sum().to_frame("missing_count"))

,filename,document_id,version,effective_date,status,classification
0,HR-BEN-005_Employee_Benefits_v4.0.pdf,HR-BEN-005,4.0,2025-07-01,Active,Internal
1,HR-COC-003_Code_of_Conduct_v5.0.pdf,HR-COC-003,5.0,2025-02-01,Active,Internal
2,HR-LV-001_Leave_Policy_v2.0.pdf,HR-LV-001,2.0,2023-04-01,Archived,Internal
3,HR-LV-001_Leave_Policy_v3.2.pdf,HR-LV-001,3.2,2025-04-01,Active,Internal
4,HR-TRV-004_Travel_Policy_v4.1.pdf,HR-TRV-004,4.1,2025-03-01,Active,Internal
5,HR-WFH-002_Work_From_Home_Policy_v2.1.pdf,HR-WFH-002,2.1,2025-01-15,Active,Internal
6,IT-DEV-002_Device_Policy_v3.0.pdf,IT-DEV-002,3.0,2025-02-15,Active,Internal
7,IT-IR-004_Incident_Response_v3.1.pdf,IT-IR-004,3.1,2025-06-01,Active,Confidential
8,IT-PWD-001_Password_Policy_v2.4.pdf,IT-PWD-001,2.4,2024-11-01,Active,Internal
9,IT-SEC-003_Security_Policy_v4.2.pdf,IT-SEC-003,4.2,2025-05-01,Active,Internal



Missing values by field:


,missing_count
document_id,0
version,0
effective_date,0
status,0
classification,0


In [ ]:
import re
from pathlib import Path
from pypdf import PdfReader

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_records = []

for _, meta in metadata_df.iterrows():
    path = next(p for p in pdf_files if p.name == meta["filename"])

    reader = PdfReader(str(path))
    raw_text = "\n".join(
        page.extract_text() or "" for page in reader.pages
    )

    # Conservative cleaning: normalize line endings and whitespace
    cleaned_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
    cleaned_text = re.sub(r"[ \t]+", " ", cleaned_text)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)
    cleaned_text = cleaned_text.strip()

    cleaned_records.append({
        **meta.to_dict(),
        "source_path": str(path.relative_to(PROJECT_ROOT)),
        "page_count": len(reader.pages),
        "raw_characters": len(raw_text),
        "cleaned_characters": len(cleaned_text),
        "cleaned_text": cleaned_text,
    })

cleaned_df = pd.DataFrame(cleaned_records)

print("Documents processed:", len(cleaned_df))
print("Empty cleaned documents:", cleaned_df["cleaned_text"].str.strip().eq("").sum())

display(
    cleaned_df[
        ["filename", "document_id", "version", "status",
         "raw_characters", "cleaned_characters"]
    ]
)

Documents processed: 13
Empty cleaned documents: 0


,filename,document_id,version,status,raw_characters,cleaned_characters
0,HR-BEN-005_Employee_Benefits_v4.0.pdf,HR-BEN-005,4.0,Active,2249,2239
1,HR-COC-003_Code_of_Conduct_v5.0.pdf,HR-COC-003,5.0,Active,2138,2127
2,HR-LV-001_Leave_Policy_v2.0.pdf,HR-LV-001,2.0,Archived,832,829
3,HR-LV-001_Leave_Policy_v3.2.pdf,HR-LV-001,3.2,Active,2201,2192
4,HR-TRV-004_Travel_Policy_v4.1.pdf,HR-TRV-004,4.1,Active,1777,1769
5,HR-WFH-002_Work_From_Home_Policy_v2.1.pdf,HR-WFH-002,2.1,Active,1886,1874
6,IT-DEV-002_Device_Policy_v3.0.pdf,IT-DEV-002,3.0,Active,1697,1686
7,IT-IR-004_Incident_Response_v3.1.pdf,IT-IR-004,3.1,Active,2474,2464
8,IT-PWD-001_Password_Policy_v2.4.pdf,IT-PWD-001,2.4,Active,1513,1506
9,IT-SEC-003_Security_Policy_v4.2.pdf,IT-SEC-003,4.2,Active,2553,2543


In [ ]:
sample = cleaned_df[
    cleaned_df["document_id"] == "HR-LV-001"
].sort_values("version")

for _, row in sample.iterrows():
    print(f"\nFile: {row['filename']}")
    print(f"Status: {row['status']}")
    print(row["cleaned_text"][:1200])
    print("-" * 70)


File: HR-LV-001_Leave_Policy_v2.0.pdf
Status: Archived
Northwind Technologies Pvt. Ltd. | HR-LV-001 Leave Policy v2.0 | Internal
Page 1
*** ARCHIVED (SUPERSEDED BY V3.2) - DO NOT USE ***
Leave Policy
Northwind Technologies Pvt. Ltd. - Human Resources Policy
Document ID
Version
Effective Date
Status
HR-LV-001
2.0
2023-04-01
Archived (Superseded by
v3.2)
Owner
Approved By
Classification
Review Cycle
Head of HR Operations
CHRO
Internal
Annual
1. Leave Entitlements
 Leave Type
Annual Entitlement
Earned Leave (EL)
15 working days
Casual Leave (CL)
7 working days
Sick Leave (SL)
8 working days
Paternity Leave
5 working days
2. Carry Forward
 2.1 A maximum of 5 unused EL days may be carried forward to the next year.
 2.2 Planned leave must be requested at least 3 days in advance.
 
3. Note
This version was superseded on 2025-04-01 by v3.2 and is retained for audit purposes only.
----------------------------------------------------------------------

File: HR-LV-001_Leave_Policy_v3.2.pdf
Stat

In [ ]:
# Create output folders
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save document metadata
metadata_df.to_csv(
    PROCESSED_DIR / "document_metadata.csv",
    index=False
)

# Save cleaned documents with metadata
cleaned_df.to_json(
    PROCESSED_DIR / "cleaned_documents.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# Save extraction quality report
quality_df.to_csv(
    REPORT_DIR / "pdf_extraction_quality.csv",
    index=False
)

# Save metadata validation report
metadata_df[
    ["filename", "document_id", "version", "effective_date",
     "status", "classification"]
].to_csv(
    REPORT_DIR / "metadata_validation_report.csv",
    index=False
)

print("Saved outputs:")
for folder in [PROCESSED_DIR, REPORT_DIR]:
    for file in sorted(folder.iterdir()):
        if file.is_file():
            print(file.name, "-", file.stat().st_size, "bytes")

Saved outputs:
cleaned_documents.jsonl - 28814 bytes
document_metadata.csv - 1093 bytes
metadata_validation_report.csv - 1093 bytes
pdf_extraction_quality.csv - 747 bytes


In [ ]:
import pandas as pd

metadata_check = pd.read_csv(
    PROCESSED_DIR / "document_metadata.csv"
)

documents_check = pd.read_json(
    PROCESSED_DIR / "cleaned_documents.jsonl",
    lines=True
)

quality_check = pd.read_csv(
    REPORT_DIR / "pdf_extraction_quality.csv"
)

assert len(metadata_check) == 13, "Metadata document count mismatch"
assert len(documents_check) == 13, "Cleaned document count mismatch"
assert documents_check["cleaned_text"].notna().all(), "Missing cleaned text"
assert documents_check["cleaned_text"].str.strip().ne("").all(), "Empty document"

print("Metadata records:", len(metadata_check))
print("Cleaned documents:", len(documents_check))
print("Quality report records:", len(quality_check))
print("All verification checks passed.")

Metadata records: 13
Cleaned documents: 13
Quality report records: 13
All verification checks passed.


# **commiting on GIT**

In [ ]:
%cd /content/secure-rag-platform-1
!git status
!git remote -v

/content/secure-rag-platform-1
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/
	data/quality_reports/

nothing added to commit but untracked files present (use "git add" to track)
origin	https://github.com/Adityasrivastava3000/secure-rag-platform-1.git (fetch)
origin	https://github.com/Adityasrivastava3000/secure-rag-platform-1.git (push)


In [ ]:
%cd /content/secure-rag-platform-1

!git add data/processed/ data/quality_reports/
!git status

/content/secure-rag-platform-1
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/processed/cleaned_documents.jsonl
	new file:   data/processed/document_metadata.csv
	new file:   data/quality_reports/metadata_validation_report.csv
	new file:   data/quality_reports/pdf_extraction_quality.csv



In [ ]:
!ls -la notebooks/

total 20
drwxr-xr-x 2 root root 4096 Oct 21 19:34 .
drwxr-xr-x 5 root root 4096 Oct 21 19:34 ..
-rw-r--r-- 1 root root    1 Oct 21 19:34 00_document_audit_cleaning.ipynb
-rw-r--r-- 1 root root    1 Oct 21 19:34 01_baseline_rag.ipynb
-rw-r--r-- 1 root root    1 Oct 21 19:34 .gitkeep


In [ ]:
%cd /content/secure-rag-platform-1

!git config --global user.name "Aditya Srivastava"
!git config --global user.email "adityasrivastava3000@gmail.com"

/content/secure-rag-platform-1


In [ ]:
!git add data/processed/ data/quality_reports/
!git add notebooks/00_document_audit_cleaning.ipynb

!git status
!git commit -m "Add document audit and cleaning pipeline"

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/processed/cleaned_documents.jsonl
	new file:   data/processed/document_metadata.csv
	new file:   data/quality_reports/metadata_validation_report.csv
	new file:   data/quality_reports/pdf_extraction_quality.csv

[main 07e2360] Add document audit and cleaning pipeline
 4 files changed, 55 insertions(+)
 create mode 100644 data/processed/cleaned_documents.jsonl
 create mode 100644 data/processed/document_metadata.csv
 create mode 100644 data/quality_reports/metadata_validation_report.csv
 create mode 100644 data/quality_reports/pdf_extraction_quality.csv


In [ ]:
import os
import subprocess
from google.colab import userdata

repo = "/content/secure-rag-platform-1"
token = userdata.get("GITHUB_TOKEN")

env = os.environ.copy()
env["GH_TOKEN_FOR_GIT"] = token

result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print("Push successful.")


fatal: could not read Username for 'https://github.com': No such device or address



In [ ]:
from google.colab import userdata

# Replace GITHUB_TOKEN with the exact name of your saved Colab Secret.
token = userdata.get("GITHUB_TOKEN")

print("GitHub token loaded:", bool(token))

GitHub token loaded: True


In [ ]:
!git add notebooks/00_document_audit_cleaning.ipynb
!git add data/processed/ data/quality_reports/

!git commit -m "Add document audit and cleaning pipeline"
!git push origin main

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
import os
import subprocess
from google.colab import userdata

repo = "/content/secure-rag-platform-1"
token = userdata.get("GITHUB_TOKEN")

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

# Supply credentials only to this push process.
askpass = "/tmp/github_askpass.sh"
with open(askpass, "w") as f:
    f.write('#!/bin/sh\n')
    f.write('case "$1" in\n')
    f.write('  *Username*) echo "x-access-token" ;;\n')
    f.write('  *Password*) echo "$GH_TOKEN_FOR_GIT" ;;\n')
    f.write('  *) exit 1 ;;\n')
    f.write('esac\n')
os.chmod(askpass, 0o700)

env["GIT_ASKPASS"] = askpass
env["GH_TOKEN_FOR_GIT"] = token

result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode == 0:
    print("Push successful.")
else:
    print(result.stderr)


Push successful.


In [24]:
%cd /content/secure-rag-platform-1
!git status
!git log -1 --oneline


/content/secure-rag-platform-1
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
07e2360 (HEAD -> main, origin/main, origin/HEAD) Add document audit and cleaning pipeline
